In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load ranked baseline (EGFR)
ranked_baseline = pd.read_parquet("../../processed/ENSG00000146648_composite_baseline.parquet")

# Load the cleaned contextual layers
signatures   = pd.read_parquet("../../processed/signatures_clean.parquet")
metabolomics = pd.read_parquet("../../processed/metabolomics_clean.parquet")
mirna        = pd.read_parquet("../../processed/mirna_clean.parquet")

print("Ranked Baseline:", ranked_baseline.shape)
print("Signatures:", signatures.shape)
print("Metabolomics:", metabolomics.shape)
print("miRNA:", mirna.shape)

Ranked Baseline: (1581, 10)
Signatures: (1955, 6)
Metabolomics: (928, 227)
miRNA: (734, 952)


In [5]:
# check how cell line IDs are stored before writing the exclusion logic
print("--- Metabolomics ---")
print("Index name:", metabolomics.index.name)
print("First 5 columns:", metabolomics.columns.tolist()[:5])

print("\n--- miRNA ---")
print("Index name:", mirna.index.name)
print("First 5 columns:", mirna.columns.tolist()[:5])

--- Metabolomics ---
Index name: None
First 5 columns: ['CCLE_ID', 'DepMap_ID', '2-aminoadipate', '3-phosphoglycerate', 'alpha-glycerophosphate']

--- miRNA ---
Index name: miRNA
First 5 columns: ['ACH-000698', 'ACH-000489', 'ACH-000431', 'ACH-000707', 'ACH-000509']


In [3]:
def apply_full_exclusion(ranked_df, sig_df, metab_df=None, mirna_df=None,
                          msi_threshold=3.0, cin_threshold=0.5,
                          exclude_metabolite=None, metabolite_threshold=None,
                          exclude_mirna=None, mirna_threshold=None):
    """
    Filters cell lines based on genomic instability (signatures) and,
    optionally, user-specified absolute thresholds for a metabolite or miRNA.

    Signatures thresholds (MSI, CIN) use established biological cutoffs.
    Metabolite/miRNA thresholds have NO built-in default — the caller must
    supply an actual value, since there is no biologically defensible
    universal cutoff for "too high" across different metabolites/miRNAs.
    """
    merged = pd.merge(ranked_df, sig_df, left_on="ACH_ID", right_index=True, how="left")

    # signatures exclusion (established thresholds)
    exclude_cond = (merged["MSIScore"] > msi_threshold) | (merged["CIN"] > cin_threshold)
    exclude_cond = exclude_cond.fillna(False)

    # metabolomics exclusion (requires explicit threshold, no default)
    if exclude_metabolite is not None:
        if metabolite_threshold is None:
            raise ValueError(
                f"exclude_metabolite='{exclude_metabolite}' was given but no "
                f"metabolite_threshold was supplied. There is no biologically "
                f"valid default — you must specify an absolute value."
            )
        if metab_df is None or exclude_metabolite not in metab_df.columns:
            raise ValueError(f"Metabolite '{exclude_metabolite}' not found in metabolomics data.")
        metab_values = metab_df.set_index("DepMap_ID")[exclude_metabolite]
        metab_high = metab_values > metabolite_threshold
        exclude_cond |= merged["ACH_ID"].map(metab_high).fillna(False)

    # miRNA exclusion (requires explicit threshold, no default)
    if exclude_mirna is not None:
        if mirna_threshold is None:
            raise ValueError(
                f"exclude_mirna='{exclude_mirna}' was given but no "
                f"mirna_threshold was supplied. There is no biologically "
                f"valid default — you must specify an absolute value."
            )
        if mirna_df is None or exclude_mirna not in mirna_df.index:
            raise ValueError(f"miRNA '{exclude_mirna}' not found in miRNA data.")
        mirna_values = mirna_df.loc[exclude_mirna]
        mirna_high = mirna_values > mirna_threshold
        exclude_cond |= merged["ACH_ID"].map(mirna_high).fillna(False)

    viable = merged[~exclude_cond].copy()
    print(f"Original: {len(merged)} | Excluded: {exclude_cond.sum()} | Remaining: {len(viable)}")
    return viable

# Run the full filter on the EGFR baseline (only utilizing signature thresholds for now)
viable_candidates = apply_full_exclusion(
    ranked_df=ranked_baseline, 
    sig_df=signatures, 
    metab_df=metabolomics, 
    mirna_df=mirna,
    msi_threshold=3.0, 
    cin_threshold=0.5
)

# Display the refined top 10
cols_to_show = ["ACH_ID", "cell_line_name", "composite_percentile", "MSIScore", "CIN", "confidence_score"]
display(viable_candidates[cols_to_show].head(10))

Original: 1581 | Excluded: 978 | Remaining: 603


,ACH_ID,cell_line_name,composite_percentile,MSIScore,CIN,confidence_score
5,ACH-002680,170MGBA,99.459094,2.18,0.478358,0.333333
6,ACH-001649,SHMAC5,99.391481,2.35,NaN,0.333333
9,ACH-000170,PRECLH,99.141832,NaN,NaN,0.625275
11,ACH-000916,NCIH1573,99.075852,0.81,0.408237,0.390583
13,ACH-000317,TUHR14TKB,98.712748,2.99,NaN,0.606448
16,ACH-000029,HCC827GR5,98.407448,2.61,NaN,0.464435
18,ACH-001442,A388,98.377282,1.91,0.225603,0.333333
22,ACH-001090,HN,97.828054,NaN,NaN,0.333333
27,ACH-000715,SNU1214,97.391689,2.21,0.300039,0.579315
28,ACH-000642,HMEL,97.380767,NaN,NaN,0.594905


In [4]:
def recommend_true_biological_twins(target_ach, viable_df, sig_df, metab_df, mirna_df, top_n=10):
    """
    Dynamically checks available omics data, builds a custom feature space, 
    and calculates similarity to find the closest viable alternative.
    """
    # Prep Signatures by moving ModelID to columns
    sig_df_clean = sig_df.reset_index()
    
    # Prep miRNA by transposing, assigning the index name, and resetting
    mirna_t = mirna_df.T
    mirna_t.index.name = "ACH_ID"
    mirna_t = mirna_t.reset_index()
    
    # Check omics data availability for the target cell line
    has_sig = target_ach in sig_df_clean["ModelID"].values
    has_metab = target_ach in metab_df["DepMap_ID"].values
    has_mirna = target_ach in mirna_t["ACH_ID"].values
    
    if not has_sig:
        return f"Cannot compute: Target {target_ach} lacks baseline signature data."
        
    print(f"--- Diagnosing Data Availability for {target_ach} ---")
    
    # Initialize the master dataframe with Signatures as the base
    master_df = sig_df_clean.copy()
    master_df = master_df.rename(columns={"ModelID": "ACH_ID"})
    active_features = [c for c in master_df.columns if c not in ['ACH_ID', 'IsDefaultEntryForModel']]
    
    # Merge additional layers if data exists for the target
    if has_metab:
        print("Target found in Metabolomics. Adding features...")
        master_df = pd.merge(master_df, metab_df, left_on="ACH_ID", right_on="DepMap_ID", how="inner")
        metab_features = [c for c in metab_df.columns if c not in ['CCLE_ID', 'DepMap_ID', 'ACH_ID']]
        active_features += metab_features
    else:
        print("Target missing Metabolomics data. Excluding from calculation.")
        
    if has_mirna:
        print("Target found in miRNA. Adding features...")
        master_df = pd.merge(master_df, mirna_t, on="ACH_ID", how="inner")
        mirna_features = [c for c in mirna_t.columns if c not in ['ACH_ID']]
        active_features += mirna_features
    else:
        print("Target missing miRNA data. Excluding from calculation.")
        
    print(f"\nFinal calculation running across {len(active_features)} valid biological dimensions...")

    # Isolate the profile and mathematical vector for the target cell line
    target_profile = master_df[master_df["ACH_ID"] == target_ach]
    target_vector = target_profile[active_features].fillna(0) 
    
    # Filter viable candidates to those surviving the inner joins
    viable_master = pd.merge(viable_df[["ACH_ID", "cell_line_name", "composite_percentile"]], master_df, on="ACH_ID", how="inner")
    
    # Remove the target cell line from the viable pool
    viable_master = viable_master[viable_master["ACH_ID"] != target_ach].copy() 
    viable_vectors = viable_master[active_features].fillna(0)
    
    if viable_vectors.empty:
         return "No viable cell lines remaining with matching omics data."

    # Calculate cosine similarity
    similarities = cosine_similarity(target_vector, viable_vectors)
    
    # Append scores and isolate top candidates
    viable_master["similarity_score"] = similarities[0]
    top_twins = viable_master.sort_values(by="similarity_score", ascending=False).head(top_n)
    
    return top_twins[["ACH_ID", "cell_line_name", "composite_percentile", "similarity_score"]]


# Execute the recommendation engine
target_cell_line = "ACH-000431" 
twins = recommend_true_biological_twins(target_cell_line, viable_candidates, signatures, metabolomics, mirna)

print(f"\nTop dynamic alternative recommendations for {target_cell_line}:")
display(twins)

--- Diagnosing Data Availability for ACH-000431 ---
Target found in Metabolomics. Adding features...
Target found in miRNA. Adding features...

Final calculation running across 965 valid biological dimensions...

Top dynamic alternative recommendations for ACH-000431:


,ACH_ID,cell_line_name,composite_percentile,similarity_score
179,ACH-000290,NCIH209,30.388792,0.938146
151,ACH-000594,DMS153,44.178239,0.934720
98,ACH-001321,TT,61.378144,0.924273
238,ACH-000382,CORL24,16.166329,0.917200
190,ACH-000743,CORL95,26.538462,0.867935
162,ACH-000790,SHP77,39.922587,0.846900
139,ACH-000830,NCIH1436,47.258543,0.767966
158,ACH-000179,NCIH1618,41.604254,0.715415
191,ACH-000780,NCIH1105,26.365788,0.685703
183,ACH-000136,CHP126,29.783638,0.681441
